# **Detector de Conflitos em Textos - PGC UFABC**

Espaço para desenvolvimento, teste e implementação do modelo de detecção de conflitos em textos, proposto no Projeto de Graduação em Computação da Universidade Federal do ABC



**Orientandos:**

*   Caio Cardoso dos Santos
*   Victor Ravazio de Lima

**Orientador:**

*   Prof. Dr. Carlos da Silva dos Santos




# **Conceitos e direções**

## **Conceiros essenciais - Super Resumo**



*   BERT - Modelo de Linguagem baseado em Transformers - Aprende relações semanticas profundas em textos

*   BERTimbau - BERT pré-treinado em portugês Br

*   Tokenizer - converte texto em numeros (IDS), onde cada inteiro é um indice no vocabulo.

  EX: "Eu adorei esse produto" => input_ids = [101, 1198, 17842, 1239, 15323, 102].
  
  Basicamento o Bert possui um dicionario estático e cada valor desse vetor de input corresponde a posição em que determinada palavra ou prefixo está localizada neste dicionario.


*   Fine-tuning - Ajustar o modelo para a tarefa especifica
*   Modelo: [neuralmind/bert-base-portuguese-cased](https://https://huggingface.co/neuralmind/bert-base-portuguese-cased)

**Fonte:** @inproceedings{souza2020bertimbau,
  author    = {F{\'a}bio Souza and
               Rodrigo Nogueira and
               Roberto Lotufo},
  title     = {{BERT}imbau: pretrained {BERT} models for {B}razilian {P}ortuguese},
  booktitle = {9th Brazilian Conference on Intelligent Systems, {BRACIS}, Rio Grande do Sul, Brazil, October 20-23 (to appear)},
  year      = {2020}
}

Fontes auxiliares e projetos:

Fonte 2: https://arthuremanuel-carosia.medium.com/usando-o-bert-para-an%C3%A1lise-de-sentimentos-363f76eee15e

Fonte 3: https://gist.github.com/Hugobsan/b507b76c7b5fc470d77b3a924211430d#file-pr-processamento-ipynb




##**Pipeline idealizado (provisório?)**


Dado o Dataset com Tweets:

*   Cada Dataset temático é divido em 3 subestruturas: Tweets genéricos, repys e quotes


  
*   Gera-se pares de tweets dentro de uma mesma subestrutura: Reply com seu respectivo replied tweet; Quote com seu respsctivo Quoted tweet; e original pareado com outro tweet origginal de forma aleatoria (amostragem aleatoria)

*   Para cada par de frases de cada grupo, constrói um dataset com essas informações: frase_1 - frase_2 - Classe - conflito (0/1)
*   Esta tabela seria o dataset de entrada para o bert - Este então aprende os padrões textuais que geralmente associam conflitos - “Que tipo de relação semântica entre frase_1 e frase_2 gera um situação de conflito 1?” -  Ex: bom x ruim; ótimo x péssimo

*   O modelo no fim é necessário pois ele aprende uma fronteira suave e não discreta.






# **Preparação do ambiente**


## **Validando GPU**

In [1]:
!nvidia-smi #verificando se o ambiente está configurado para GPU


Wed May 20 16:17:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## **Instalando bibliotecas**

**Sobre as bibliotecas:**


*   Transformers (coleção de modelos pré-treinados em Transformers)
*   datasets (fornece alguns datasets otimizados prontos)
*   accelerate (gerencia multi gpu/otimiza treino em gpu)
*   evaluate (biblioteca de metricas padronizadas)

*   PyTorch (framework de deep learning)

* As 4 primeiras pertencem ao Hugging Face, espécie de "GitHub" de modelos IA







In [2]:
#Instalando:
!pip install -q transformers datasets torch accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [3]:
# modulo para importar dataset a partir de um link
!pip install gdown

# **Importações**

## **Importando Bibliotecas**

**Sobre as bibliotecas:**

*  Torch (Framework para aplicações de machine learn: abstrai tarefas, operacionaliza manipulação de tensores e etc)

*  BertTokenizer(Tokenizar textos)
*   BertModel (Bert puro)
*   BertForSequenceClassification (BERT + Camada de classificador)
*   Trainer (modulo para treino)

*   TrainigArguments (modulo para treino)

*   Gdown (baixar arquivos direto do drive)

* Dataset (otimiza e facilita a formatação do datataset para treino no Bert)

* sklearn (biblioteca que fornece ferramentas prontas para realizar tarefas de ML, como por exemplo "train_test_split" que divide automaticamente treino e teste, ou as metricas de avaliação que abstrai os calculos de precisão, recall e etc)






In [4]:
import torch
import numpy as np
import pandas as pd
import gdown

from transformers import (
    BertTokenizer,
    BertModel,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score


## **Importando o Modelo**

In [5]:
#Tokenizer
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

#Modelo: BERT + Camada de classificação
#num_labels corresponde ao numero de classes do modelo (Conflito/não conflito)
model = BertForSequenceClassification.from_pretrained("neuralmind/bert-base-portuguese-cased", num_labels = 2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

## **Importando o dataset**

In [6]:
# baixando dataset para o collab
file_id = "1eGuTbLr6W5crwM5PKlDt8Rab1xNRUYnX"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "cloroquina_replys.csv", quiet=False)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Downloading...
From: https://drive.google.com/uc?id=1eGuTbLr6W5crwM5PKlDt8Rab1xNRUYnX
To: /content/cloroquina_replys.csv

100%|██████████| 2.37M/2.37M [00:00<00:00, 109MB/s]


'cloroquina_replys.csv'

In [7]:
#carregando o dataset
df = pd.read_csv ("cloroquina_replys.csv")
df = df.iloc[:100]
df = df[df['Conflict'] != -1] #eliminando indeterminados
df.shape

(100, 4)

In [8]:
#Verificando distribuição das classes
print("Quantidade de anotações por classes:")
print("0 - Não Conflito: ", df[df["Conflict"] == 0].shape[0])
print("1 - Conflito: ", df[df["Conflict"] == 1].shape[0])

Quantidade de anotações por classes:
0 - Não Conflito:  56
1 - Conflito:  44


# **Modelo - Conflict Detector**

## **Separando treino e teste**

In [9]:
#Separando dados em treino e teste
# train_test_split é um método da biblioteca sklearn que abstrai a divisão de treino e test
# O parametro stratify mantém a proporção das classes - Originalmente 80% não conflito e 20% não conflito, assim também será em train e test.
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Conflict"])

#convertendo para o formato "datasets" do Hugging face - Abstrai alguns procedimentos, como formatar para tensores e etc
train_dataset = Dataset.from_pandas(train_df, preserve_index=False) #index = false - não inclui os indices do df original
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)


In [10]:
train_dataset

Dataset({
    features: ['tweet_1', 'tweet_2', 'Class', 'Conflict'],
    num_rows: 80
})

## **Tokenização**

1 - Tokenização consiste em transformar o texto bruto em uma representação numérica estruturada que o modelo BERT consiga processar. Para isso, o texto é dividido em subpalavras (subword tokenization), e cada token é convertido em um identificador numérico correspondente ao índice no vocabulário aprendido pelo modelo.

O processo de tokenização gera três principais componentes:

- input_ids: sequência de identificadores numéricos dos tokens.
- token_type_ids: indica a qual sentença cada token pertence (no caso de pares de sentenças).
- attention_mask: indica quais posições são tokens reais (1) e quais são padding (0).

Na abordagem adotada neste trabalho, a tokenização é feita em pares de tweets, explorando a arquitetura do BERT, que permite modelar relações entre duas sentenças simultaneamente. Dessa forma, o modelo aprende interações semânticas entre os dois textos, em vez de processá-los isoladamente.

A entrada do BERT segue o formato:

[CLS] texto_1 [SEP] texto_2 [SEP]

Para construir essa entrada, utiliza-se o tokenizer do BERT, passando ambos os textos como parâmetros, além de argumentos de controle como padding (para padronizar o tamanho das sequências) e truncation (para limitar o tamanho máximo).

Esses vetores tokenizados são então utilizados como entrada do modelo, que gera embeddings contextualizados para cada token.

Observação: o padding é necessário para garantir que todas as sequências de um batch tenham o mesmo tamanho, já que o processamento é realizado em lotes (batches). Isso é essencial para a eficiência computacional e compatibilidade com o formato esperado pelo modelo.

2 - Este processo é realizado através da função tokenize_function, que aplica o tokenizer do BERT (carregado na etapa de importação) em cada elemento do dataset. No caso deste trabalho, a tokenização é aplicada a pares de tweets.


3 - O metodo "map" a seguir, que pertence a biblioteca "datasets" do Hugging Face, aplica tokenize_function em cada entrada do df e faz o merge do resultado com as colunas originais
EX:
    # originaal:'tweet_1', 'tweet_2', 'Class', 'Conflict;
    # Tokenize: 'input_ids', 'token_type_ids', 'attention_mask'
    # Map: 'tweet_1', 'tweet_2', 'Class', 'Conflict', 'input_ids', 'token_type_ids', 'attention_mask'
     
Naturalmente "train_dataset" e "test_dataset" é um objeto "Dataset" (da biblioteca do hugging face)

In [11]:
#Função de tokenização
def tokenize_function(pair):

    return tokenizer(
        pair["tweet_1"],
        pair["tweet_2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [12]:
#Aplicando a Tokenização
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [13]:
#Formato pós tokenização
train_dataset

Dataset({
    features: ['tweet_1', 'tweet_2', 'Class', 'Conflict', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 80
})

In [14]:
#Exemplo
print(train_dataset[0])

{'tweet_1': 'O protocolo da cloroquina é o "foda-se" institucionalizado.', 'tweet_2': 'O que fazer se o médico entender que aquele paciente não tem indicação de cloroquina? Prescreve mesmo assim?', 'Class': 1, 'Conflict': 0.0, 'input_ids': [101, 231, 14491, 180, 20139, 11198, 253, 146, 107, 227, 285, 118, 176, 107, 19237, 2303, 119, 102, 231, 179, 1434, 176, 146, 4714, 9050, 179, 6086, 10268, 346, 376, 9403, 125, 20139, 11198, 136, 2311, 1128, 301, 653, 1016, 136, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
# Renomeando o atributo alvo - O modulo trainer espera o atributo alvo numa coluna "label"
train_dataset = train_dataset.rename_column("Conflict", "labels")
test_dataset = test_dataset.rename_column("Conflict", "labels")
train_dataset

Dataset({
    features: ['tweet_1', 'tweet_2', 'Class', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 80
})

## **Formatando Tensores**

O Bert trabalha com tensores PyTorch, sendo necessária a conversão dos dados para este formato antes do treinamento.

Após a tokenização, os dados ainda estão em estruturas Python comuns (listas e inteiros). Para resolver isso, a biblioteca Datasets é bastante conveniente, pois o método set_format configura o dataset para que, no momento do acesso aos elementos (de forma lazy), as colunas especificadas sejam automaticamente convertidas para torch.Tensor.

O uso da biblioteca Datasets abstrai a necessidade de implementar manualmente uma classe "Dataset" do PyTorch (com métodos como __getitem__ e __len__), simplificando significativamente o pipeline de pré-processamento.

Pós-tokenização:
dataset[0] retorna:

"input_ids": [101, 2023, 2003],
"attention_mask": [1, 1, 1],
"labels": 1

Pós-formatação (set_format):
dataset[0] retorna:

"input_ids": tensor([...]),
"attention_mask": tensor([...]),
"labels": tensor(...)

     

In [16]:
# Formatando os Tensores:

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "token_type_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "token_type_ids", "attention_mask", "labels"]
)